# Tutorial 2 — Running ecCount

This notebook runs the released ecCount model on one image, step by step, with the
same preprocessing and post-processing as `python -m ecdna_bench.cli.run_eccount`,
and then on a folder of your own images.

**Weights.** Download `eccount_best.pt` from the repository's GitHub release and
either place it at `release/model_checkpoints/eccount_best.pt` or set
`ECCOUNT_WEIGHTS` to its path (see `docs/TUTORIAL_EXTERNAL.md`, section 4).
Without weights the notebook still runs, with an untrained network, so that you can
check the installation; its counts are meaningless.

**Hardware.** A GPU is optional. On a CPU one image takes several seconds.

In [ ]:
import os, sys, time
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "configs" / "default.yaml").exists():
            return p
    raise RuntimeError("Run this notebook from inside the ecdna-bench repository.")

REPO = find_repo_root()
# Where you downloaded the BioImage Archive files (the folder that contains images/).
DATA_ROOT = Path(os.environ.get("ECDNA_DATA_ROOT", Path.home() / "ecdna_data")).expanduser()
if (DATA_ROOT / "Files" / "images").is_dir():
    DATA_ROOT = DATA_ROOT / "Files"
HAVE_DATA = (DATA_ROOT / "images" / "gt_image").is_dir()
print("repository :", REPO)
print("data folder:", DATA_ROOT, "(found)" if HAVE_DATA else "(not found: the notebook runs on a small synthetic example)")

try:
    import ecdna_bench
    print("ecdna_bench:", Path(ecdna_bench.__file__).parent)
except ImportError:
    sys.path.insert(0, str(REPO / "src"))
    import ecdna_bench
    print("ecdna_bench imported from", REPO / "src")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

NATIVE_SHAPE = (2048, 2448)
ANCHOR_UID = "ncih2170_facs_fish_0723_low_her2_52"   # the example image used in the paper

SUFFIXES = ("", "_pred_roi", "_predicted_roi", "_pred", "_roi", "_mask")

def find_file(folder, uid):
    """The file in `folder` named after the unique identifier (any image extension)."""
    folder = Path(folder) if folder else None
    if folder is None or not folder.is_dir():
        return None
    for suffix in SUFFIXES:
        hits = sorted(p for p in folder.rglob(uid + suffix + ".*")
                      if p.stem == uid + suffix
                      and p.suffix.lower() in {".tif", ".tiff", ".png", ".npy", ".npz"})
        if hits:
            return hits[0]
    return None

def read_image(path, color=False):
    flag = cv2.IMREAD_COLOR if color else cv2.IMREAD_UNCHANGED
    img = cv2.imread(str(path), flag)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    if color:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else np.dstack([img] * 3)
    elif img.ndim == 3:
        img = img.max(axis=2)
    return img

def render_diamonds(points_rc, shape, radius=2):
    """Render (row, col) points as diamonds (|dy| + |dx| <= radius; 13 px for radius 2)."""
    mask = np.zeros(shape, np.uint8)
    offsets = [(dy, dx) for dy in range(-radius, radius + 1)
               for dx in range(-radius, radius + 1) if abs(dy) + abs(dx) <= radius]
    for r, c in np.asarray(points_rc, dtype=int):
        for dy, dx in offsets:
            y, x = r + dy, c + dx
            if 0 <= y < shape[0] and 0 <= x < shape[1]:
                mask[y, x] = 255
    return mask

def synthetic_example(seed=0, n=60):
    """A small stand-in image set, used only when the data folder is missing."""
    rng = np.random.default_rng(seed)
    pts = np.stack([rng.integers(800, 1250, n), rng.integers(1000, 1450, n)], 1)
    gs = render_diamonds(pts, NATIVE_SHAPE)
    rgb = np.full(NATIVE_SHAPE + (3,), 8, np.uint8)
    for r, c in pts:
        cv2.circle(rgb, (int(c), int(r)), 2, (40, 220, 60), -1)
    roi = np.zeros(NATIVE_SHAPE, np.uint8); roi[700:1350, 900:1550] = 255
    return {"uid": "synthetic_example", "rgb": rgb, "dapi": rgb[:, :, 2].copy(),
            "gs": gs, "roi": roi, "points": pts}

def load_image_set(uid=None):
    """Load RGB, DAPI, gold-standard mask, points and ROI for one image set."""
    if not HAVE_DATA:
        return synthetic_example()
    img_dir = DATA_ROOT / "images"
    if uid is None:
        uid = ANCHOR_UID if find_file(img_dir / "gt_image", ANCHOR_UID) else \
            sorted(p.stem for p in (img_dir / "gt_image").iterdir())[0]
    out = {"uid": uid}
    for key, sub, color in [("rgb", "rgb", True), ("dapi", "dapi", False),
                            ("gs", "gt_image", False), ("roi", "roi_mask", False)]:
        p = find_file(img_dir / sub, uid)
        out[key] = read_image(p, color=color) if p else None
    p = find_file(img_dir / "gt_coords", uid)
    out["points"] = np.load(p, allow_pickle=True) if p else None
    return out

## 1. Settings from the released configuration

In [ ]:
import yaml
import torch

with open(REPO / "configs" / "default.yaml") as fh:
    CFG = yaml.safe_load(fh)
ecc = CFG["eccount"]
INPUT_H, INPUT_W = ecc["input_size"]                  # 1024 x 1224: images are downsampled by two
PP = ecc["postprocess"]
print("input size:", INPUT_H, "x", INPUT_W)
print("post-processing:", PP)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Build the network and load the weights

In [ ]:
from ecdna_bench.eccount.model import ModelConfig, build_model
from ecdna_bench.eccount.infer import InferConfig, infer_one, load_checkpoint
from ecdna_bench.eccount.postprocess import PostprocessConfig, peaks_to_mask

model = build_model(ModelConfig())
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")          # 7,849,601 for the published architecture

weights = Path(os.environ.get("ECCOUNT_WEIGHTS", REPO / "release" / "model_checkpoints" / "eccount_best.pt"))
if weights.is_file():
    info = load_checkpoint(str(weights), model, device=DEVICE)
    print("loaded", weights, "| best epoch", info.get("best_epoch"), "| best val loss", info.get("best_val_loss"))
    TRAINED = True
else:
    print("WARNING: no weights at", weights, "- continuing with an UNTRAINED network (counts are meaningless)")
    TRAINED = False
model.eval().to(DEVICE)

infer_cfg = InferConfig(
    postprocess=PostprocessConfig(
        smooth_sigma=float(PP["smooth_sigma"]), reapply_roi=bool(PP["reapply_roi"]),
        threshold_abs=float(PP["threshold_abs"]), peak_min_distance=int(PP["peak_min_distance"]),
        nms_min_distance=int(PP["nms_min_distance"]), exclude_border=int(PP["exclude_border"]),
        point_disk_radius=int(PP["point_disk_radius"])),
    threshold_mask_cutoff=0.5,
    peaks_disk_radius=int(PP["point_disk_radius"]),
)

## 3. One image, step by step

The function below is the per-image body of `run_eccount`: downsample by two
(area interpolation), apply the ROI, divide by 255, run the network, find peaks, and
map the peak coordinates back to the native 2,448 x 2,048 px frame, where each peak
is drawn as a 5 x 5-pixel diamond like the gold-standard points.

In [ ]:
def run_eccount_on(rgb, roi=None):
    orig_h, orig_w = rgb.shape[:2]
    x = cv2.resize(rgb, (INPUT_W, INPUT_H), interpolation=cv2.INTER_AREA)
    roi_small = None
    if roi is not None:
        roi_small = (cv2.resize(roi, (INPUT_W, INPUT_H), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
        x = x * roi_small[:, :, None]
    t = torch.from_numpy(x.astype(np.float32).transpose(2, 0, 1) / 255.0).unsqueeze(0).to(DEVICE)
    res = infer_one(t, roi_small, model, infer_cfg)
    sx, sy = orig_w / INPUT_W, orig_h / INPUT_H
    peaks = [(int(round(px * sx)), int(round(py * sy)), s) for px, py, s in res.peaks]
    peaks_mask = peaks_to_mask(peaks, shape=(orig_h, orig_w), disk_radius=infer_cfg.peaks_disk_radius)
    thr_mask = cv2.resize(res.threshold_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return res.prob_map, peaks, peaks_mask, thr_mask

sample = load_image_set()
t0 = time.time()
prob, peaks, peaks_mask, thr_mask = run_eccount_on(sample["rgb"], sample["roi"])
print(f"{sample['uid']}: {len(peaks)} peaks in {time.time() - t0:.1f} s")

In [ ]:
from ecdna_bench.evaluation.objects import objects_from_mask

def count_objects(mask):
    # counting needs no per-object masks, which keeps this fast
    return len(objects_from_mask(mask, min_area=3, connectivity=8, attach_mask=False))

n_peaks_objects = count_objects(peaks_mask)
n_thr_objects = count_objects(thr_mask)
n_gs = count_objects(sample["gs"]) if sample["gs"] is not None else None
print("ecCount (peaks)          objects:", n_peaks_objects)
print("ecCount (threshold mask) objects:", n_thr_objects)
print("gold standard            objects:", n_gs)
if not TRAINED:
    print("(untrained network: these counts are not meaningful)")

In [ ]:
ys, xs = np.nonzero(sample["gs"] if sample["gs"] is not None else peaks_mask)
if len(ys) == 0:
    ys, xs = np.array([NATIVE_SHAPE[0] // 2]), np.array([NATIVE_SHAPE[1] // 2])
y0, y1 = max(0, ys.min() - 40), min(NATIVE_SHAPE[0], ys.max() + 40)
x0, x1 = max(0, xs.min() - 40), min(NATIVE_SHAPE[1], xs.max() + 40)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(sample["rgb"][y0:y1, x0:x1]); axes[0].set_title("input (crop)")
axes[1].imshow(cv2.resize(prob, (NATIVE_SHAPE[1], NATIVE_SHAPE[0]))[y0:y1, x0:x1], cmap="magma")
axes[1].set_title("probability map")
view = sample["rgb"][y0:y1, x0:x1].copy()
view[peaks_mask[y0:y1, x0:x1] > 0] = (0, 255, 255)
if sample["gs"] is not None:
    view[(sample["gs"][y0:y1, x0:x1] > 0) & (peaks_mask[y0:y1, x0:x1] == 0)] = (255, 0, 255)
axes[2].imshow(view); axes[2].set_title("peaks (cyan) and gold standard (magenta)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 4. A folder of your own images

Images should be probe-channel RGB composites acquired like the resource
(x60, 2,448 x 2,048 px). Other sizes are resized to the network input, which changes
the apparent size of ecDNA signals; check a few results by eye before relying on
them. An ROI mask with the same file name restricts the analysis to one metaphase
spread; without it the whole field is analyzed.

For many images, the command-line route is faster:
`python scripts/prepare_local_run.py images --images <folder> --out-dir runs/my_images`
(see `NEXT_STEPS.txt` in the output folder).

In [ ]:
import pandas as pd

MY_IMAGES = Path(os.environ.get("MY_IMAGES", DATA_ROOT / "images" / "rgb"))   # change to your folder
MY_ROIS = Path(os.environ["MY_ROIS"]) if os.environ.get("MY_ROIS") else None   # e.g. Path("my_rois"): binary masks named like the images
OUT = REPO / "runs" / "tutorial_02"
MAX_IMAGES = 3            # remove the limit for a real run

rows = []
files = sorted(p for p in MY_IMAGES.glob("*") if p.suffix.lower() in {".tif", ".tiff", ".png", ".jpg"})[:MAX_IMAGES] \
    if MY_IMAGES.is_dir() else []
if not TRAINED:
    # An untrained network marks most pixels as candidate peaks; whole images then take very long.
    print("skipped: this section needs the released weights (see section 2)")
    files = []
elif not files:
    print("no images found in", MY_IMAGES)
for p in files:
    rgb = read_image(p, color=True)
    roi_path = find_file(MY_ROIS, p.stem) if MY_ROIS else None
    roi = read_image(roi_path) if roi_path else None
    _, pk, pk_mask, _ = run_eccount_on(rgb, roi)
    (OUT / "peaks").mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(OUT / "peaks" / f"{p.stem}.png"), pk_mask)
    rows.append({"image": p.name, "height": rgb.shape[0], "width": rgb.shape[1],
                 "roi": bool(roi_path), "ecdna_count": count_objects(pk_mask)})
table = pd.DataFrame(rows)
if len(table):
    OUT.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUT / "ecdna_counts.csv", index=False)
    print("written:", OUT / "ecdna_counts.csv")
table